# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1: The Anatomy of Growing Content

The paper reports that growing content tends to be longer, younger, and slightly better positioned than declining content. The authors also note that this is an observational comparison rather than evidence of causation.

**Methodology question:** How was the "growing" versus "declining" label created, and was every feature measured before that label was assigned? If the label is derived from recent impression changes, I would want to confirm that no feature contains overlapping information from the same time period.

---

### Finding 2: The Content Performance Curve

The paper reports that content performs best between 61 and 90 days, declines after about 270 days, and that refreshed older pages can recover.

**Methodology question:** Does the validation design separate naturally aging content from content that was actively refreshed? A grouped or time-aware evaluation would help determine whether the observed pattern reflects genuine content lifecycle effects rather than differences between content groups or update timing.

In [17]:
paper_findings = [
    "The Anatomy of Growing Content",
    "The Content Performance Curve"
]

print("Paper findings reviewed:")
for finding in paper_findings:
    print("-", finding)

print(f"\nTotal findings reviewed: {len(paper_findings)}")

Paper findings reviewed:
- The Anatomy of Growing Content
- The Content Performance Curve

Total findings reviewed: 2


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


To audit my Week 5 model, I evaluated the same Random Forest model using both a standard random train/test split and a grouped train/test split based on `client_id`.

The random split achieved higher performance, with an Accuracy of **0.8675** and an F1-score of **0.8814**. Using the grouped split, the model achieved an Accuracy of **0.8192** and an F1-score of **0.8315**.

The grouped split provides a more realistic evaluation because it tests the model on clients that were not seen during training. The lower performance suggests that the random split benefits from similarities between content from the same client. Based on this comparison, I consider the grouped split to provide a more trustworthy estimate of how the model would perform on unseen client data.

In [18]:
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

df = pd.read_csv("content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

groups = df["client_id"]

X = df.drop(columns=[
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
])

y = df["is_declining_label"]

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])


X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf.fit(X_train_r, y_train_r)

pred_r = rf.predict(X_test_r)


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

rf.fit(X_train_g, y_train_g)

pred_g = rf.predict(X_test_g)

results = pd.DataFrame({
    "Validation": [
        "Random Split",
        "Grouped Split"
    ],
    "Accuracy": [
        accuracy_score(y_test_r, pred_r),
        accuracy_score(y_test_g, pred_g),
    ],
    "Precision": [
        precision_score(y_test_r, pred_r),
        precision_score(y_test_g, pred_g),
    ],
    "Recall": [
        recall_score(y_test_r, pred_r),
        recall_score(y_test_g, pred_g),
    ],
    "F1": [
        f1_score(y_test_r, pred_r),
        f1_score(y_test_g, pred_g),
    ],
})

results

,Validation,Accuracy,Precision,Recall,F1
0,Random Split,0.867500,0.855778,0.908672,0.881432
1,Grouped Split,0.819244,0.793990,0.872658,0.831467


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I reviewed the final feature set to identify potential sources of data leakage before training the model.

The primary leakage risk was the use of `trend_direction` and `trend_pct`, since the target label (`is_declining_label`) was derived from the same trend information. Including these columns would allow the model to learn the answer directly instead of learning useful patterns, so both columns were removed before training.

I also excluded identifier columns such as `content_id` and `client_id` from the feature set. The grouped train/test split based on `client_id` further reduced the risk of information leaking between related content from the same client.

After these checks, the remaining features consisted of historical content and performance metrics that were available before prediction. Based on this audit, I did not identify any remaining direct sources of label leakage in the final model.

In [19]:
excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used",
]

print("Excluded columns:")
for column in excluded_columns:
    print("-", column)

print("\nRemaining feature count:", X.shape[1])
print("Target column:", "is_declining_label")

Excluded columns:
- content_id
- client_id
- trend_direction
- trend_pct
- provider_used
- model_used

Remaining feature count: 38
Target column: is_declining_label


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**Original claim:**

"Random Forest accurately predicts declining content."

**Rewritten claim:**

In this dataset, the Random Forest model achieved the strongest measured performance among the evaluated models. Using a grouped train/test split, it achieved an Accuracy of **0.8192** and an F1-score of **0.8315**. These results suggest that the model can support decisions about identifying declining content, but they do not demonstrate that it will generalize to every dataset or future content without additional validation.

In [20]:
claim_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Score"],
    "Grouped Split": [0.819244, 0.831467]
})

claim_metrics

,Metric,Grouped Split
0,Accuracy,0.819244
1,F1 Score,0.831467


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.